# calc_03c. ERA5: observed change (calculations)

Part 2 of the ERA5 analysis, over the years ERA5 and LESFMIP share (1979–2013). Part 1, notebooks calc_03a and calc_03b, asked whether ERA5 could be one more member of each model's historical ensemble. Attribution asks more of the observations:

1. **Detection.** Could the observed change have happened without human influence? ERA5 outside the **hist-nat** members' range says no.
2. **Consistency.** Which experiments could have produced it: historical, or a single forcing on its own?
3. **Records.** With no change, year *n* is its record's warmest so far with probability 1/*n*. Warming brings more record highs and fewer record lows.
4. **Scaling factors.** How much of each model's forced response is in the observations.

Mostly in regional means (`oc.REGIONS`), which average away much of a grid point's noise.

This notebook works everything out and saves it in `RESULTS_DIR` (`results/tas/sample` or `results/tas/full` under `SCRATCH_DIR`), beside calc_03a's and calc_03b's results; notebook plot_01d opens them and draws the figures. It needs only notebook calc_01's seasonal means, so it can run before or after calc_03a and calc_03b.

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from extant import datatree, era5_evaluation as ev, observed_change as oc, preprocessing as pp
from extant.config import MAIN_EXPERIMENTS, SATELLITE_YEAR_RANGE, VARIABLES
from extant.datatree import skip_empty, tree_to_dataset
from extant.logger import setup_logging
from extant.plots import charts, forced_response as fr, methods
from plotting_modules.constants import SEASONS

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

#(t): Harmless warnings, silenced the same way in every notebook. zarr's, that parts of the stores (the text
#(t): coordinates, the consolidated metadata) aren't in the Zarr 3 standard yet:
warnings.filterwarnings('ignore', message='.*Zarr (format 3|V3) specification')
#(t): numpy's, about slices with nothing but NaN (land, a model without a member) or nothing at all, too few members for
#(t): a standard deviation, and dividing by zero or by NaN (the log of a probability of 0, say):
warnings.filterwarnings('ignore', message='All-NaN slice encountered', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Mean of empty slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Degrees of freedom <= 0 for slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='(divide by zero|invalid value) encountered', category=RuntimeWarning)
#(t): dask's, about large task graphs, which are large on purpose:
warnings.filterwarnings('ignore', message='.*Sending large graph of size', category=UserWarning)
#(t): Every figure's text at the notebook's sizes and colours (fr.NOTEBOOK), as in every notebook
plt.rcParams.update(fr.NOTEBOOK.rc)

In [ ]:
%matplotlib inline

## Settings

The experiments (`MAIN_EXPERIMENTS`) and the years ERA5 and LESFMIP share (`SATELLITE_YEAR_RANGE`) are in `extant/config.py`, and the order of the seasons (`SEASONS`) in `plotting_modules/constants.py`, the same for every notebook. Only this notebook's own choices are here.

In [ ]:
#(t): The variable: its name and its units once notebook calc_01 has converted it (VARIABLE.name, VARIABLE.units)
VARIABLE = VARIABLES['tas']

In [ ]:
#(t): The grid point of the tests at one point (the 2.5° cell nearest config.POINT, on the Antarctic plateau)
SAMPLE_LAT_LON = dict(lat=-82.5, lon=120.0)

#(t): The model of the one-model quick looks
EXAMPLE_MODEL = 'CanESM5'

In [ ]:
#(t): The region of the one-region figures (the model of the one-model figures is EXAMPLE_MODEL)
OBS_REGION = 'Antarctica (90–65°S)'
oc.REGIONS

In [ ]:
#(t): Where notebook calc_01's seasonal means are, and where this notebook's results go: JASMIN's scratch
SCRATCH_DIR = Path('/work/scratch-pw4/aborow/ExtAnt')
SCRATCH_DIR

## Open the Data

Run **one** of the next two sections. Both leave `lesfmip_season_tree` (seasonal means, /<model>/<experiment>, on member × year × season × lat × lon), `era5_seasonal_da` (ERA5 on year × season × lat × lon, on the same grid), all in °C, and `RESULTS_DIR`, where the results are saved.

### Sample data

The 3×3 block of grid cells around the point in `tests/data` (two models), put through notebook calc_01's pre-processing.

In [ ]:
#(t): The sample's files, in this repository: <model>_<experiment>_sample.nc, monthly tas in K on (member, time, lat, lon)
SAMPLE_DIR = Path('../tests/data')
sample_files = sorted(SAMPLE_DIR.glob('*_*_sample.nc'))
sample_files

In [ ]:
#(t): The model and experiment of each file, from its name
sample_names = [path.stem.removesuffix('_sample').rsplit('_', 1) for path in sample_files]

In [ ]:
#(t): Each file of the experiments analysed opened, as a DataTree /<model>/<experiment>
lesfmip_monthly_tree = xr.DataTree.from_dict({f'{model}/{experiment}': xr.open_dataset(path).load()
                                              for path, (model, experiment) in zip(sample_files, sample_names)
                                              if experiment in MAIN_EXPERIMENTS})

In [ ]:
#(t): Seasonal means in °C, through notebook calc_01's pre-processing (pp.seasonal_tree: to °C, cut at 2014-12, complete
#(t): seasons in whole years)
lesfmip_season_tree = pp.seasonal_tree(lesfmip_monthly_tree, VARIABLE.name).compute()

In [ ]:
#(t): ERA5's sample file: monthly tas in K from 1979, on (time, lat, lon)
era5_sample_path = SAMPLE_DIR / 'era5_sample.nc'
era5_monthly_da = xr.open_dataarray(era5_sample_path).load()
era5_sample_path

In [ ]:
#(t): Its seasonal means in °C, on the models' grid (pp.seasonal_era5, as in notebook calc_01)
era5_seasonal_da = pp.seasonal_era5(era5_monthly_da, VARIABLE.name, like=lesfmip_season_tree)

In [ ]:
#(t): Where the sample's results are saved
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'sample'
RESULTS_DIR

### Full data

The seasonal means notebook calc_01 saved (section 3) in `SCRATCH_DIR`. Opening is lazy.

In [ ]:
#(t): The two stores notebook calc_01 saved
seasonal_path = SCRATCH_DIR / 'seasonal' / f'lesfmip_{VARIABLE.name}.zarr'
era5_path = SCRATCH_DIR / 'seasonal' / f'era5_{VARIABLE.name}.zarr'
seasonal_path, era5_path

In [ ]:
#(t): Every model and experiment notebook calc_01 saved
lesfmip_saved_tree = xr.open_datatree(seasonal_path, engine='zarr', chunks={})

In [ ]:
#(t): Only the experiments analysed (filter keeps the model nodes above them)
lesfmip_season_tree = lesfmip_saved_tree.filter(lambda node: node.name in MAIN_EXPERIMENTS)

In [ ]:
#(t): ERA5's seasonal means
era5_seasonal_da = xr.open_zarr(era5_path)[VARIABLE.name].load()

In [ ]:
#(t): Where the full data's results are saved
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'full'
RESULTS_DIR

### What was opened

Whichever section ran. An empty square is a model and experiment that was never opened.

In [ ]:
#(t): Members per model (rows) and experiment (columns)
member_counts_df = pd.Series({(node.parent.name, node.name): node.sizes['member']
                              for node in lesfmip_season_tree.leaves}).unstack()
member_counts_df = member_counts_df.reindex(columns=MAIN_EXPERIMENTS)

In [ ]:
#(t): The members: each model (rows) and experiment (columns), grey where there are none
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=1, n_legends=0, n_colorbars=1,
                                                  panel_width=0.9 * member_counts_df.shape[1] + 0.5,
                                                  panel_height=0.4 * member_counts_df.shape[0] + 0.3, left_margin=1.8),
                 layout='constrained')
axes = fig.add_gridspec(1, 1).subplots(squeeze=False)
counts_image = methods.plot_member_counts(member_counts_df, ax=axes[0, 0])
fig.colorbar(counts_image, ax=axes, location='bottom', shrink=0.6, label='Members')
fig.suptitle('LESFMIP members per model and experiment', x=0.01, ha='left');

In [ ]:
#(t): The experiments every model has
OBS_EXPERIMENTS = [experiment for experiment in MAIN_EXPERIMENTS
                   if all(experiment in model_node.children for model_node in lesfmip_season_tree.children.values())]
OBS_EXPERIMENTS

## Data over ERA5's Years

What the sections below use, made as calc_03a makes it (in *The Data* and *Regional Evaluation*), so that this notebook needs nothing from calc_03a: LESFMIP over `SATELLITE_YEAR_RANGE`, each model's members and ERA5 at the point, and the regional means of ERA5 and of every member.

In [ ]:
#(t): LESFMIP over the years shared with ERA5
lesfmip_obs_tree = lesfmip_season_tree.sel(year=SATELLITE_YEAR_RANGE)

In [ ]:
#(t): Each model's historical members at the point, on (member, year, season)
historical_point_tree = lesfmip_obs_tree.match('*/historical').sel(SAMPLE_LAT_LON).load()

In [ ]:
#(t): ERA5 at the point over the same years, on (year, season)
era5_point_da = era5_seasonal_da.sel(year=SATELLITE_YEAR_RANGE).sel(SAMPLE_LAT_LON)

In [ ]:
#(t): Each model's hist-nat members at the point too, for the record rates there (section 2)
histnat_point_tree = lesfmip_obs_tree.match('*/hist-nat').sel(SAMPLE_LAT_LON).load()

In [ ]:
#(t): ERA5's regional means, on (region, year, season)
era5_regional_da = oc.regional_means(era5_seasonal_da).rename(VARIABLE.name)

In [ ]:
%%time
#(t): Every member's regional means over the ERA5 years, every model and experiment
regional_obs_tree = oc.regional_means(lesfmip_obs_tree).compute()
regional_obs_tree

## 1. Consistency with Each Experiment

ERA5's 1979–2013 trend, and its S/N against itself, among each experiment's members, in `OBS_REGION`. ERA5 outside **hist-nat** is a detection of forced change; ERA5 inside **historical** is consistency.

In [ ]:
#(t): Every member's regional mean in OBS_REGION, every model and experiment (regional_obs_tree, above)
obs_region_tree = regional_obs_tree.sel(region=OBS_REGION)

In [ ]:
#(t): ERA5's, over SATELLITE_YEAR_RANGE
era5_obs_region_da = era5_regional_da.sel(region=OBS_REGION, year=SATELLITE_YEAR_RANGE)

In [ ]:
#(t): Quick look: EXAMPLE_MODEL's historical (grey) and hist-nat (green) members, and ERA5 (black), in JJA
obs_region_tree[EXAMPLE_MODEL]['historical']['tas'].sel(season='JJA').plot.line(x='year', hue='member',
                                                                                add_legend=False,
                                                                             color='0.7', lw=0.5)
obs_region_tree[EXAMPLE_MODEL]['hist-nat']['tas'].sel(season='JJA').plot.line(x='year', hue='member', add_legend=False,
                                                                           color='tab:green', lw=0.3)
era5_obs_region_da.sel(season='JJA').plot(color='k', lw=2);

In [ ]:
#(t): The trend of every member
region_trend_tree = obs_region_tree.map_over_datasets(skip_empty(ev.trend))

In [ ]:
#(t): ... and ERA5's
era5_region_trend_da = ev.trend(era5_obs_region_da)

In [ ]:
#(t): Where ERA5's falls among each model's members
region_trend_test_tree = xr.DataTree.from_dict({
    node.path: ev.locate(node['tas'], era5_region_trend_da) for node in region_trend_tree.leaves
})

In [ ]:
#(t): The S/N against itself, the same way: every member's
region_emergence_tree = obs_region_tree.map_over_datasets(skip_empty(ev.self_signal_to_noise))

In [ ]:
#(t): ERA5's
era5_region_emergence_da = ev.self_signal_to_noise(era5_obs_region_da)

In [ ]:
#(t): Where ERA5's falls among each experiment's members
region_emergence_test_tree = xr.DataTree.from_dict({
    node.path: ev.locate(node['tas'], era5_region_emergence_da) for node in region_emergence_tree.leaves
})

In [ ]:
#(t): Each experiment's two tests along a statistic dim
region_tests_tree = xr.DataTree.from_dict({
    node.path: xr.concat([node.to_dataset(), region_emergence_test_tree[node.path].to_dataset()],
                         dim=xr.Variable('statistic', ['trend', 'emergence']))
    for node in region_trend_test_tree.leaves
})
region_tests_tree

In [ ]:
#(t): Quick look: ERA5's trend percentile among each experiment's members, every model
region_trend_verdicts_tree = xr.DataTree.from_dict({node.path: node.to_dataset()[['percentile', 'verdict', 'testable']]
                                                    for node in region_trend_test_tree.leaves})
region_trend_verdicts_ds = tree_to_dataset(region_trend_verdicts_tree, dims=['model', 'experiment'])
region_trend_verdicts_ds = region_trend_verdicts_ds.assign_coords(model=region_trend_verdicts_ds['model'].astype(str),
                                                                  experiment=region_trend_verdicts_ds['experiment'].astype(str))
region_trend_verdicts_ds['percentile'].sel(season=SEASONS).plot(x='season', y='model', col='experiment', vmin=0,
                                                                vmax=100, cmap='RdBu_r');

In [ ]:
#(t): Saved for plot_01d
region_tests_path = RESULTS_DIR / 'region_tests.zarr'
region_tests_tree.to_zarr(region_tests_path, mode='w')
region_tests_path

## 2. Record Rates

Cumulative record highs and lows through 1979–2013, the four seasons pooled so there are enough records to compare, for ERA5 and the hist-nat and historical members (`oc.record_curves`).

In [ ]:
#(t): In OBS_REGION
record_curves_ds = oc.record_curves({'hist-nat': obs_region_tree[EXAMPLE_MODEL]['hist-nat']['tas'],
                                     'historical': obs_region_tree[EXAMPLE_MODEL]['historical']['tas']},
                                    era5_obs_region_da)
record_curves_ds

In [ ]:
#(t): At the point (much noisier than a regional mean)
point_curves_ds = oc.record_curves({'hist-nat': histnat_point_tree[EXAMPLE_MODEL]['hist-nat']['tas'],
                                    'historical': historical_point_tree[EXAMPLE_MODEL]['historical']['tas']},
                                   era5_point_da)

In [ ]:
#(t): Quick look: the record highs, ERA5 (black) and each historical member (grey)
record_curves_ds['members'].sel(experiment='historical', kind='highs').plot.line(x='year', hue='member',
                                                                                 add_legend=False, color='0.7')
record_curves_ds['obs'].sel(kind='highs').plot(color='k');

In [ ]:
#(t): Saved for plot_01d
record_curves_path = RESULTS_DIR / 'record_curves.zarr'
record_curves_ds.to_zarr(record_curves_path, mode='w')
record_curves_path

In [ ]:
#(t): Saved for plot_01d
point_record_curves_path = RESULTS_DIR / 'point_record_curves.zarr'
point_curves_ds.to_zarr(point_record_curves_path, mode='w')
point_record_curves_path

## 3. Detection Maps

The trend test at every grid point, against hist-nat and against historical, sorted into four classes (`oc.DETECTION_CLASSES`): detected and consistent (outside hist-nat, inside historical), not detected and consistent, detected and inconsistent, not detected and inconsistent. It takes a few minutes, so it saves its result.

### Calculate (slow; saves the result)

In [ ]:
#(t): ERA5's trend among one experiment's members at every grid point (one model's experiment loaded at a time;
#(t): the members dropped)
def map_trend_test(ds):
    """Where ERA5's 1979-2013 trend falls among one experiment's members' trends, at every grid point.

    Args:
        ds (xr.Dataset): One model's experiment: tas on (member, year, season, lat, lon), over SATELLITE_YEAR_RANGE.

    Returns:
        xr.Dataset: ev.locate's results (without the members) on (season, lat, lon).
    """
    members_trend_da = ev.trend(ds['tas'].load())
    era5_trend_da = ev.trend(era5_seasonal_da.sel(year=SATELLITE_YEAR_RANGE))
    return ev.locate(members_trend_da, era5_trend_da).drop_dims('member')

In [ ]:
%%time
#(t): Example: EXAMPLE_MODEL's historical
example_trend_map_ds = map_trend_test(lesfmip_obs_tree[EXAMPLE_MODEL]['historical'].to_dataset())
example_trend_map_ds

In [ ]:
#(t): The trend test at the point, as plot_01b makes it from calc_03a's trends: ERA5's trend among EXAMPLE_MODEL's
#(t): historical members'
point_trend_test_ds = ev.locate(ev.trend(historical_point_tree[EXAMPLE_MODEL]['historical']['tas']),
                                ev.trend(era5_point_da))

In [ ]:
#(t): Check: at the point, the map gives the point's trend test (largest difference of ERA5's percentile: 0)
float(abs(example_trend_map_ds['percentile'].sel(SAMPLE_LAT_LON) - point_trend_test_ds['percentile']).max())

In [ ]:
%%time
#(t): For historical and hist-nat of every model, on (model, experiment, season, lat, lon)
detection_tree = lesfmip_obs_tree.filter(lambda node: node.name in ('historical', 'hist-nat'))
map_trend_tests_ds = datatree.reduce_to_dataset(detection_tree, map_trend_test)

In [ ]:
#(t): Each grid point's class (quick look: EXAMPLE_MODEL's, every season)
detection_class_da = oc.detection_class(map_trend_tests_ds.sel(experiment='historical'),
                                        map_trend_tests_ds.sel(experiment='hist-nat'))
detection_class_da.sel(model=EXAMPLE_MODEL).plot(x='lon', y='lat', col='season');

In [ ]:
#(t): The classes and ERA5's percentile among each experiment's members, as notebook plot_06 opens them
detections_ds = xr.Dataset({
    'detection_class': detection_class_da,
    'historical_percentile': map_trend_tests_ds['percentile'].sel(experiment='historical', drop=True),
    'hist-nat_percentile': map_trend_tests_ds['percentile'].sel(experiment='hist-nat', drop=True),
})
detections_ds

In [ ]:
#(t): Saved for plot_01d and notebook plot_06
detections_path = RESULTS_DIR / 'detections.zarr'
detections_ds.to_zarr(detections_path, mode='w')
detections_path

## 4. Scaling Factors

A regression of ERA5 on each model's forced responses, over every year and season of 1979–2014, in each region (`oc.scaling_factors`): ANT + NAT (anthropogenic, historical − hist-nat, and natural), and GHG + AER + O3 + NAT. The uncertainty comes from a perfect-model test: each historical member in turn plays the observations, regressed on fingerprints that leave it out.

**How `oc.scaling_factors` works**, step by step for EXAMPLE_MODEL in `OBS_REGION`, ANT + NAT: how much of the model's anthropogenic (ANT) and natural (NAT) responses ERA5 holds, and how sure that is.

In [ ]:
#(t): EXAMPLE_MODEL's historical and hist-nat members in OBS_REGION, and ERA5's, over oc.YEARS
scaling_historical_da = regional_obs_tree[EXAMPLE_MODEL]['historical']['tas'].sel(region=OBS_REGION).sel(year=oc.YEARS)
scaling_hist_nat_da = regional_obs_tree[EXAMPLE_MODEL]['hist-nat']['tas'].sel(region=OBS_REGION).sel(year=oc.YEARS)
scaling_era5_da = era5_regional_da.sel(region=OBS_REGION).sel(year=oc.YEARS)

In [ ]:
#(t): The whole method in one call, keeping each member's perfect-model β, to check each step against
example_scaling_ds = oc.scaling_factors({'historical': regional_obs_tree[EXAMPLE_MODEL]['historical']['tas'].sel(region=OBS_REGION),
                                         'hist-nat': regional_obs_tree[EXAMPLE_MODEL]['hist-nat']['tas'].sel(region=OBS_REGION)},
                                        era5_regional_da.sel(region=OBS_REGION), 'ANT + NAT')

In [ ]:
#(t): Step 1: only the samples (a year and a season) that ERA5 and every member have
scaling_valid_da = (scaling_era5_da.notnull() & scaling_historical_da.notnull().all('member')
                    & scaling_hist_nat_da.notnull().all('member'))

In [ ]:
#(t): ... each series as anomalies from its own mean in each season, so only the changes through the years are fitted
scaling_era5_anomaly_da = oc._anomalies(scaling_era5_da.where(scaling_valid_da))
scaling_historical_anomaly_da = oc._anomalies(scaling_historical_da.where(scaling_valid_da))
scaling_hist_nat_anomaly_da = oc._anomalies(scaling_hist_nat_da.where(scaling_valid_da))

In [ ]:
#(t): Step 2: the fingerprints, ensemble-mean anomalies: ANT is historical's minus hist-nat's, NAT is hist-nat's
scaling_ant_da = scaling_historical_anomaly_da.mean('member') - scaling_hist_nat_anomaly_da.mean('member')
scaling_nat_da = scaling_hist_nat_anomaly_da.mean('member')

In [ ]:
#(t): Step 3: ERA5 ≈ β_ANT × ANT + β_NAT × NAT, by least squares over every year and season
# Example: if ERA5's anomalies were exactly 2 × ANT + 0.5 × NAT in every sample, the least-squares β would be [2, 0.5].
# Each series becomes one vector, year by year with the seasons inside, and the fingerprints are its columns; samples
# with anything missing are left out (oc._least_squares).
scaling_X_arr = np.column_stack([scaling_ant_da.transpose('year', 'season').values.ravel(),
                                 scaling_nat_da.transpose('year', 'season').values.ravel()])
scaling_y_arr = scaling_era5_anomaly_da.transpose('year', 'season').values.ravel()
scaling_beta_arr = oc._least_squares(scaling_y_arr, scaling_X_arr)

In [ ]:
#(t): Step 4, how sure: each historical member in turn plays ERA5, against fingerprints made without it. Member 0's β:
# A member is the model's own response plus internal variability, so its β is about 1, and how far it is from 1 is what
# internal variability alone does to β. Leaving it out of the fingerprint keeps its own noise out of the fit.
scaling_left_out_ant_da = (scaling_historical_anomaly_da.drop_isel(member=0).mean('member')
                           - scaling_hist_nat_anomaly_da.mean('member'))
scaling_left_out_X_arr = np.column_stack([scaling_left_out_ant_da.transpose('year', 'season').values.ravel(),
                                          scaling_nat_da.transpose('year', 'season').values.ravel()])
scaling_member_0_beta_arr = oc._least_squares(
    scaling_historical_anomaly_da.isel(member=0).transpose('year', 'season').values.ravel(), scaling_left_out_X_arr)

In [ ]:
#(t): ERA5's interval: its β minus the 5-95% range of every member's β about their median
# Example: the members' β_ANT have a median of 1.0 and run from 0.4 to 1.7 (5% to 95%), deviations of -0.6 to +0.7.
# ERA5's β_ANT of 1.5 then has the interval 1.5 - 0.7 = 0.8 to 1.5 + 0.6 = 2.1. It excludes 0, so ANT is detected, and
# it includes 1, so the model's response has the right size.
scaling_pseudo_beta_arr = example_scaling_ds['pseudo_beta'].values
scaling_centre_arr = np.median(scaling_pseudo_beta_arr, axis=0)
scaling_spread_low_arr, scaling_spread_high_arr = np.quantile(scaling_pseudo_beta_arr - scaling_centre_arr,
                                                              [ev.ALPHA / 2, 1 - ev.ALPHA / 2], axis=0)
scaling_lower_arr = scaling_beta_arr - scaling_spread_high_arr
scaling_upper_arr = scaling_beta_arr - scaling_spread_low_arr

In [ ]:
#(t): Check: the steps against oc.scaling_factors: ERA5's β, member 0's, and ERA5's interval
pd.DataFrame({'step by step': [*scaling_beta_arr, *scaling_member_0_beta_arr, *scaling_lower_arr, *scaling_upper_arr],
              'oc.scaling_factors': [*example_scaling_ds['beta'].values,
                                     *example_scaling_ds['pseudo_beta'].isel(member=0).values,
                                     *example_scaling_ds['lower'].values, *example_scaling_ds['upper'].values]},
             index=[f'{name} {signal}' for name in ['beta', 'member 0 beta', 'lower', 'upper'] for signal in ['ANT', 'NAT']])

In [ ]:
#(t): The scaling factors step by step: a) each season, ERA5's anomalies (blue) and the fit, β_ANT × ANT + β_NAT × NAT
#(t): (black); b) each signal, every member's β (grey), their median (dashed), and ERA5's β with its interval (blue)
scaling_fit_da = scaling_beta_arr[0] * scaling_ant_da + scaling_beta_arr[1] * scaling_nat_da
fig, axes = plt.subplot_mosaic([SEASONS, ['ANT', 'ANT', 'NAT', 'NAT']], figsize=(16, 7), layout='constrained')
for season in SEASONS:
    axes[season].plot(scaling_era5_anomaly_da['year'], scaling_era5_anomaly_da.sel(season=season), color='blue')
    axes[season].plot(scaling_fit_da['year'], scaling_fit_da.sel(season=season), color='k')
    axes[season].set(title=f'a) {season}: ERA5 (blue) and the fit (black)', ylabel=VARIABLE.units)
for index, signal in enumerate(['ANT', 'NAT']):
    axes[signal].hist(scaling_pseudo_beta_arr[:, index], bins=20, color='0.7')
    axes[signal].axvline(scaling_centre_arr[index], color='k', ls='--')
    axes[signal].axvspan(scaling_lower_arr[index], scaling_upper_arr[index], color='blue', alpha=0.15)
    axes[signal].axvline(scaling_beta_arr[index], color='blue')
    axes[signal].set(title=f'b) β_{signal}: every member (grey), their median (dashed), ERA5 and its interval (blue)',
                     xlabel='β', ylabel='members');

In [ ]:
%%time
#(t): ANT + NAT (oc.SIGNAL_SETS): every model's scaling factors in each region, as a DataTree /<model>/<region>. Each
#(t): model's historical node names the model, whose experiments oc.scaling_factors takes from regional_obs_tree. The
#(t): members' perfect-model betas are dropped
scaling_ant_nat_tree = xr.DataTree.from_dict({
    f'{node.parent.name}/{region}': oc.scaling_factors(
        {experiment: regional_obs_tree[node.parent.name][experiment]['tas'].sel(region=region)
         for experiment in OBS_EXPERIMENTS},
        era5_regional_da.sel(region=region), 'ANT + NAT').drop_vars('pseudo_beta')
    for node in regional_obs_tree.match('*/historical').leaves for region in oc.REGIONS
})

In [ ]:
#(t): On (model, region, signal), as plot_01d opens it (the names as text, not Python objects)
scaling_ant_nat_ds = tree_to_dataset(scaling_ant_nat_tree, dims=['model', 'region'])
scaling_ant_nat_ds = scaling_ant_nat_ds.assign_coords(model=scaling_ant_nat_ds['model'].astype(str),
                                                      region=scaling_ant_nat_ds['region'].astype(str))
scaling_ant_nat_ds

In [ ]:
%%time
#(t): The same for GHG + AER + O3 + NAT
scaling_single_tree = xr.DataTree.from_dict({
    f'{node.parent.name}/{region}': oc.scaling_factors(
        {experiment: regional_obs_tree[node.parent.name][experiment]['tas'].sel(region=region)
         for experiment in OBS_EXPERIMENTS},
        era5_regional_da.sel(region=region), 'GHG + AER + O3 + NAT').drop_vars('pseudo_beta')
    for node in regional_obs_tree.match('*/historical').leaves for region in oc.REGIONS
})

In [ ]:
scaling_single_ds = tree_to_dataset(scaling_single_tree, dims=['model', 'region'])
scaling_single_ds = scaling_single_ds.assign_coords(model=scaling_single_ds['model'].astype(str),
                                                    region=scaling_single_ds['region'].astype(str))
scaling_single_ds

In [ ]:
#(t): Saved for plot_01d
scaling_ant_nat_path = RESULTS_DIR / 'scaling_ant_nat.zarr'
scaling_ant_nat_ds.to_zarr(scaling_ant_nat_path, mode='w')
scaling_ant_nat_path

In [ ]:
#(t): Saved for plot_01d
scaling_single_forcings_path = RESULTS_DIR / 'scaling_single_forcings.zarr'
scaling_single_ds.to_zarr(scaling_single_forcings_path, mode='w')
scaling_single_forcings_path